# **Algiers in 24 Hours - Tourist Optimizer**

---
# **SECTION 0 - Imports and Global Configuration**

In [ ]:
TMAX_FULL = 720
TMAX_HALF = 360
STARTING_TIME = 482

---
# **SECTION 1 - Shared Data Structures**
### All 6 Members

## 1.1 - Location Class

In [ ]:
class Landmark:
    """
    Represents a single tourist landmark or attraction.
    
    Attributes:
        id (int): A unique numerical identifier for the landmark.
        name (str): The name of the landmark.
        lon (float): Longitude coordinate.
        lat (float): Latitude coordinate.
        interest_score (float): The rating or user interest score.
        opening_hours (dict[str, list[int]]): A dictionary mapping days of the week 
            (e.g., 'Monday') to a list of 24 integers representing hours. 
            1 means open, 0 means closed. Example: {'Monday': [0,0,...,1,1,0]}
        visit_duration (int): Estimated time spent at the landmark in minutes.
        landmark_type (str): The category of the landmark (e.g., 'Museum', 'Park').
    """

    def __init__(self, id: int, name: str, lon: float, lat: float, 
                 interest_score: float, opening_hours: dict[str, list[int]], 
                 visit_duration: int, landmark_type: str):
        
        self.id = id
        self.name = name
        self.lon = lon
        self.lat = lat
        self.interest_score = interest_score
        self.opening_hours = opening_hours
        self.visit_duration = visit_duration
        self.landmark_type = landmark_type

    def is_open(self, day: str, hour: int) -> bool:
        """
        Checks if the landmark remains open for the entire duration of a planned visit.
        
        Args:
            day (str): The day of the week to check (e.g., 'Monday').
            hour (int): The starting hour of the visit (0-23).
            
        Returns:
            bool: True if the landmark is open for every hour of the visit, False otherwise.
        """
        # Retrieve the 24-hour schedule for the specified day
        opening = self.opening_hours.get(day)
        
        # Failsafe: if the day doesn't exist in the dictionary, assume it's closed
        if not opening:
            return False

        # Calculate how many hours the visit will take (rounded up to be safe)
        duration_in_hours = int(self.visit_duration / 60)
        
        # Check every hour from the arrival time to the departure time
        for i in range(hour, hour + duration_in_hours + 1):
            
            # i % 24 ensures that hour 24 wraps back to 0 (midnight)
            if opening[i % 24] == 0:
                return False 

        return True

In [ ]:
#this class represent single hotel

class Hotel:

    def __init__(self , name , lon, lat ):
      self.id = name
      self.lon = lon
      self.lat = lat 



## 1.2 - Problem Class

## Class for informed search algorithms

In [ ]:
# This class holds everything the algorithms need and give clean access to it.

## Class for local search algorithms


In [ ]:
# state is path of Nodes

## 1.3 - Solution Class

In [ ]:
# This class store the result of an algorithm and validate it.

---
# **SECTION 2 - Data Collection**

## 2.1 - Landmarks Dataset


We gathered - clean - data about  **+60 attractions/landmarks in Algiers**, including the following attributes:  
* __Attraction/landmark name.__
* __Description (__ short description __)__  
* __Type of attraction/landmark (__ Monument, Park/Garden, Historical site ... etc __)__  
* __Rating (/10)__ 
* __GPS coordinates (__ longitude, latitude __)__
* __Estimated visit duration (__ in minutes  __)__
* __Opening Hours__
* __Images__


In [3]:
import csv
from collections import Counter

# path to the landmarks CSV file
LANDMARK_PATH = "./../dataset/landmarks/Algiers_landmarks.csv"

with open(LANDMARK_PATH, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

ratings = [float(r['Rating']) for r in rows]
types = Counter(r['Type'] for r in rows)


print(f"Total landmarks : {len(rows)}")
print(f"Average rating  : {sum(ratings)/len(ratings):.2f}")
print(f"Highest rating  : {max(ratings)}")
print(f"Lowest rating   : {min(ratings)}")
print(f"Unique types    : {len(types)}")


print("\nCount by type: ")
for t, c in types.most_common():
    print(f"  {t:<25} {c}")


print("\nTop 10 landmarks: ")
for i, r in enumerate(sorted(rows, key=lambda x: float(x['Rating']), reverse=True)[:10], 1):
    print(f"  {i:>2}. {r['Rating']}  {r['Name']}")

print("\nBottom 5 landmarks: ")
for i, r in enumerate(sorted(rows, key=lambda x: float(x['Rating']))[:5], 1):
    print(f"  {i}. {r['Rating']}  {r['Name']}")

Total landmarks : 62
Average rating  : 7.50
Highest rating  : 9.7
Lowest rating   : 3.2
Unique types    : 15

Count by type: 
  Historical Site           11
  Mosque                    8
  Museum                    7
  Park/Garden               6
  Cultural Center           6
  Square/Plaza              5
  Palace                    4
  Monument                  3
  Beach                     3
  Shopping/Mall             3
  Park/Forest               2
  Cathedral                 1
  Park/Promenade            1
  Marina/Coastal            1
  Beach/Coastal             1

Top 10 landmarks: 
   1. 9.7  Great Mosque of Algiers (Djamaa el Djazaïr)
   2. 9.6  Botanical Garden Hamma
   3. 9.5  Martyrs' Memorial (Maqam E'chahid)
   4. 9.5  Notre Dame d'Afrique
   5. 9.3  La Grande Poste
   6. 9.2  Palais des Raïs (Bastion 23)
   7. 9.1  Ketchaoua Mosque
   8. 9.0  National Museum of Fine Arts
   9. 9.0  Algiers Opera House
  10. 8.9  Bardo National Museum

Bottom 5 landmarks: 
  1. 3.2  Ben A

## 2.2 - Hotels Dataset


We gathered a dataset of __+180 hotels__ across the Algiers region, collected from Google Maps.

In [ ]:
HOTELS_PATH = "./../dataset/hotels/Algiers_hotels.csv"

with open(HOTELS_PATH, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
 
print(f"Total hotels: {len(rows)}")

Total hotels: 184


## 2.2 - Time Matrix


Generated via the OSRM public API (router.project-osrm.org) using the driving profile.

- `time_matrix.npy` — (247 × 247) numpy array of travel times in minutes
- `time_matrix_named.csv` — same matrix with landmark/hotel names as row and column headers

Rows and columns are ordered: landmarks first (83), hotels second (184).

**Note:** The public OSRM API enforces a 100-coordinate limit per request,
so the matrix is built in batches. Results reflect road-network driving time
and may not account for real-time traffic.

In [ ]:
import pandas as pd
import numpy as np
import requests

landmarks = pd.read_csv("./../dataset/landmarks/final_dataset.csv")
hotels    = pd.read_csv("./../dataset/hotels/Algiers_hotels.csv")

names = list(landmarks["Name"]) + list(hotels["name"])

all_coords = (
    [(row["Longitude"], row["Latitude"]) for _, row in landmarks.iterrows()] +
    [(row["longitude"], row["latitude"]) for _, row in hotels.iterrows()]
)

OSRM_BASE = "http://router.project-osrm.org/table/v1/driving"
BATCH     = 100  # public OSRM hard limit

n = len(all_coords)
time_matrix = np.zeros((n, n), dtype=float)

for i in range(0, n, BATCH):
    for j in range(0, n, BATCH):
        src_slice = all_coords[i:i+BATCH]
        dst_slice = all_coords[j:j+BATCH]

        combined = src_slice + dst_slice
        coords_str = ";".join(f"{lon},{lat}" for lon, lat in combined)

        sources = ";".join(str(k) for k in range(len(src_slice)))
        destinations = ";".join(str(k + len(src_slice)) for k in range(len(dst_slice)))

        r = requests.get(
            f"{OSRM_BASE}/{coords_str}",
            params={"annotations": "duration", "sources": sources, "destinations": destinations},
            timeout=120
        )
        r.raise_for_status()
        block = np.array(r.json()["durations"], dtype=float) / 60  # seconds → minutes

        time_matrix[i:i+len(src_slice), j:j+len(dst_slice)] = block

np.save("time_matrix.npy", time_matrix)

df = pd.DataFrame(time_matrix.round(1), index=names, columns=names)
df.to_csv("time_matrix_named.csv")

print(f"Done. Matrix shape: {time_matrix.shape}")

The CSV file `time_matrix_named.csv` is converted into a JSON file `time_matrix.json`

In [ ]:
import pandas as pd
import json

# load the existing matrix CSV
df = pd.read_csv("time_matrix_named.csv", index_col=0)

# convert to nested dict {start: {end: minutes}}
time_dict = {}
for start in df.index:
    time_dict[start] = {}
    for end in df.columns:
        time_dict[start][end] = round(float(df.loc[start, end]), 1)

# save as JSON
with open("time_matrix.json", "w", encoding="utf-8") as f:
    json.dump(time_dict, f, ensure_ascii=False, indent=2)

print("Done. Saved to time_matrix.json")

---
# **SECTION 3 - Algorithms Representation**


## Greedy , Simulated Anealing , GA + optional Ones


code impelmentation + definition of each algorithm and of each parameter


---
# **SECTION 4 - Testing Data on ALgorithms**

---
# **SECTION 6 - Final Results and Submission**
### Person 1 assembles - All review

## 6.1 - Master Comparison Table

In [ ]:
# TODO

## 6.2 - All Visualizations

In [ ]:
# TODO

## 6.3 - Conclusions